In [1]:
import os
# Prevent memory fragmentation
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg19, VGG19_Weights
from torchvision.utils import save_image
from PIL import Image
import random
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

class DenseResidualBlock(nn.Module):
    def __init__(self, filters=64, res_scale=0.2):
        super().__init__()
        self.res_scale = res_scale
        self.conv1 = nn.Conv2d(filters, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(filters + 32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(filters + 64, 32, 3, 1, 1)
        self.conv4 = nn.Conv2d(filters + 96, 32, 3, 1, 1)
        self.conv5 = nn.Conv2d(filters + 128, filters, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        c1 = self.lrelu(self.conv1(x))
        c2 = self.lrelu(self.conv2(torch.cat((x, c1), 1)))
        c3 = self.lrelu(self.conv3(torch.cat((x, c1, c2), 1)))
        c4 = self.lrelu(self.conv4(torch.cat((x, c1, c2, c3), 1)))
        c5 = self.conv5(torch.cat((x, c1, c2, c3, c4), 1))
        return c5.mul(self.res_scale) + x

class RRDB(nn.Module):
    def __init__(self, filters=64):
        super().__init__()
        self.rdb1 = DenseResidualBlock(filters)
        self.rdb2 = DenseResidualBlock(filters)
        self.rdb3 = DenseResidualBlock(filters)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out.mul(0.2) + x

class RRDBNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, filters=64, num_rrdb=8):
        super().__init__()
        self.conv_first = nn.Conv2d(in_channels, filters, 3, 1, 1)
        self.rrdb_blocks = nn.Sequential(*[RRDB(filters) for _ in range(num_rrdb)])
        self.conv_trunk = nn.Conv2d(filters, filters, 3, 1, 1)
        
        self.upsample1 = nn.Sequential(nn.Conv2d(filters, filters * 4, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, inplace=True))
        self.upsample2 = nn.Sequential(nn.Conv2d(filters, filters * 4, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, inplace=True))
        
        self.conv_last = nn.Sequential(
            nn.Conv2d(filters, filters, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(filters, out_channels, 3, 1, 1)
        )

    def forward(self, x):
        feat = self.conv_first(x)
        trunk = self.conv_trunk(self.rrdb_blocks(feat))
        feat = feat + trunk
        feat = self.upsample1(feat) 
        feat = self.upsample2(feat) 
        return torch.clamp(self.conv_last(feat), 0, 1)


In [2]:
# ==============================
# SETTINGS
# ==============================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/kaggle/input/datasets/whyamanbhardwaj/inference/RRDB_4x_epoch_20.pth"
TEST_DIR = "/kaggle/input/datasets/thang1703/celebahq-1024x1024/CelebAHQ-1024x1024"
SAVE_DIR = "/kaggle/working/test_outputs"

HR_SIZE = 256
LR_SIZE = 64
BATCH_SIZE = 16
NUM_WORKERS = 2

os.makedirs(SAVE_DIR, exist_ok=True)

print("Using device:", DEVICE)

# ==============================
# LOAD MODEL
# ==============================
model = RRDBNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ==============================
# DATASET (SAME AS TRAINING)
# ==============================
class TestDataset(Dataset):
    def __init__(self, root_dir):
        self.files = [
            os.path.join(root_dir, f)
            for f in os.listdir(root_dir)
            if f.lower().endswith(("png", "jpg", "jpeg"))
        ]

        self.transform = transforms.Compose([
            transforms.RandomCrop(HR_SIZE, pad_if_needed=True),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        return self.transform(img), os.path.basename(self.files[idx])


dataset = TestDataset(TEST_DIR)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ==============================
# PSNR FUNCTION
# ==============================
def calculate_psnr(sr, hr):
    mse = torch.mean((sr - hr) ** 2, dim=[1,2,3])
    psnr = 20 * torch.log10(torch.tensor(1.0, device=sr.device)) - 10 * torch.log10(mse)
    return psnr

# ==============================
# TEST LOOP
# ==============================
total_psnr = 0
total_images = 0

with torch.no_grad():
    loop = tqdm(loader)

    for hr, filenames in loop:
        hr = hr.to(DEVICE)

        # SAME degradation as training
        lr = F.interpolate(hr, size=(LR_SIZE, LR_SIZE), mode='bicubic', align_corners=False)
        noise = torch.randn_like(lr) * random.uniform(0.01, 0.05)
        lr = torch.clamp(lr + noise, 0, 1)

        with torch.amp.autocast("cuda"):
            sr = model(lr)

        batch_psnr = calculate_psnr(sr, hr)

        total_psnr += batch_psnr.sum().item()
        total_images += hr.size(0)

        # Save comparison images
        lr_up = F.interpolate(lr, size=(HR_SIZE, HR_SIZE), mode='bicubic')
        comparison = torch.cat([lr_up, sr, hr], dim=3)

        for i in range(comparison.size(0)):
            save_image(
                comparison[i],
                os.path.join(SAVE_DIR, f"comparison_{filenames[i]}")
            )

        loop.set_postfix(PSNR=f"{batch_psnr.mean().item():.2f}")

avg_psnr = total_psnr / total_images
print("\n🔥 Final Average PSNR:", round(avg_psnr, 2), "dB")

Using device: cuda


100%|██████████| 1875/1875 [07:06<00:00,  4.39it/s, PSNR=32.12]


🔥 Final Average PSNR: 31.78 dB
